# 07 — Candidate Fusion + DCN Ranker — MerRec

Notebook này dùng 5 retrieval model:

- Content-Based
- Co-Visitation
- ALS
- GRU4Rec
- Two-Tower

Pipeline:

`Retrieval → Candidate Export → Candidate Fusion → Feature Engineering → DCN → Top-K`

DCN chỉ làm **ranking**, không retrieve trực tiếp từ catalog 27M item.

## 0. Điều kiện đầu vào bắt buộc

Thư mục:

`D:\MerRec\candidate_exports`

Cần 15 file:

### TRAIN
- `content_train.parquet`
- `covis_train.parquet`
- `als_train.parquet`
- `gru_train.parquet`
- `twotower_train.parquet`

### VAL
- `content_val.parquet`
- `covis_val.parquet`
- `als_val.parquet`
- `gru_val.parquet`
- `twotower_val.parquet`

### TEST
- `content_test.parquet`
- `covis_test.parquet`
- `als_test.parquet`
- `gru_test.parquet`
- `twotower_test.parquet`

Mỗi file phải có:

`case_id | user_id | candidate_item_id | target_item_id | score | rank`

Tất cả model phải dùng cùng định nghĩa `case_id` và cùng target next-item.

In [ ]:
# Nếu máy thieu thư viện:
# !pip install torch polars pyarrow numpy pandas scikit-learn

## 1. Import

In [ ]:
from pathlib import Path
import os, sys, json, math, random, hashlib
import numpy as np
import pandas as pd
import polars as pl
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import roc_auc_score

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Torch:", torch.__version__)
print("Device:", DEVICE)
print("CPU threads:", os.cpu_count())


## 2. Paths + Top-K

In [ ]:
PROJECT_ROOT = Path(r"D:\MerRec")

MODEL_DATA = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "recommender"
    / "model_data"
)

CANDIDATE_DIR = PROJECT_ROOT / "candidate_exports"
DCN_ROOT = PROJECT_ROOT / "training" / "dcn"
DATASET_DIR = DCN_ROOT / "dataset"
MODEL_DIR = DCN_ROOT / "model"
METRIC_DIR = DCN_ROOT / "metrics"

for d in [CANDIDATE_DIR, DATASET_DIR, MODEL_DIR, METRIC_DIR]:
    d.mkdir(parents=True, exist_ok=True)

SOURCE_TOPK = {
    "content": 150,
    "covis": 250,
    "als": 200,
    "gru": 200,
    "twotower": 300,
}

TRAIN_PATH = MODEL_DATA / "interactions" / "split=train" / "data_0.parquet"

print("MODEL_DATA:", MODEL_DATA)
print("CANDIDATE_DIR:", CANDIDATE_DIR)
print("DCN_ROOT:", DCN_ROOT)

print("\\nTop-K:")
for k, v in SOURCE_TOPK.items():
    print(k, "=", v)

## 3. Kiểm tra candidate exports

In [ ]:
def candidate_path(source, split):
    return CANDIDATE_DIR / f"{source}_{split}.parquet"

all_ok = True

for split in ["train", "val", "test"]:
    print("\n" + "=" * 55)
    print("SPLIT:", split)

    for source in SOURCE_TOPK:
        p = candidate_path(source, split)
        ok = p.exists()
        all_ok = all_ok and ok
        print(f"{source:10s}", "[OK]" if ok else "[FAIL]", p)

if not all_ok:
    print("\n[WARN] Chua du candidate files. Dừng tại đây và export candidate từ 5 retrieval model trước.")
else:
    print("\n[OK] Da du candidate files.")

## 4. Load và chuẩn hóa candidate

Giữ:
- raw score
- score normalize theo từng case
- rank
- reciprocal rank

In [ ]:
def load_candidate(source, split, topk):
    p = candidate_path(source, split)
    if not p.exists():
        raise FileNotFoundError(p)

    df = pl.read_parquet(p)

    required = {
        "case_id", "user_id", "candidate_item_id",
        "target_item_id", "score", "rank"
    }

    missing = required - set(df.columns)
    if missing:
        raise ValueError(f"{p} thieu columns: {sorted(missing)}")

    df = (
        df.select([
            pl.col("case_id").cast(pl.Utf8),
            pl.col("user_id").cast(pl.Utf8),
            pl.col("candidate_item_id").cast(pl.Utf8),
            pl.col("target_item_id").cast(pl.Utf8),
            pl.col("score").cast(pl.Float32),
            pl.col("rank").cast(pl.Int32),
        ])
        .filter(pl.col("rank") <= topk)
        .unique(subset=["case_id", "candidate_item_id"], keep="first")
    )

    df = (
        df
        .with_columns([
            pl.col("score").min().over("case_id").alias("_mn"),
            pl.col("score").max().over("case_id").alias("_mx"),
        ])
        .with_columns([
            (
                (pl.col("score") - pl.col("_mn"))
                / (pl.col("_mx") - pl.col("_mn") + 1e-8)
            ).cast(pl.Float32).alias("score_norm"),
            (1.0 / (pl.col("rank").cast(pl.Float32) + 1.0)).alias("rr"),
        ])
        .drop(["_mn", "_mx"])
        .rename({
            "score": f"{source}_score",
            "score_norm": f"{source}_score_norm",
            "rank": f"{source}_rank",
            "rr": f"{source}_rr",
        })
    )

    print(
        f"{source:10s} {split:5s}: "
        f"{df.height:,} rows | {df['case_id'].n_unique():,} cases"
    )
    return df


## 5. Candidate Fusion

In [ ]:
def build_candidate_fusion(split):
    frames = [
        load_candidate(source, split, topk)
        for source, topk in SOURCE_TOPK.items()
    ]

    keys = ["case_id", "user_id", "candidate_item_id", "target_item_id"]
    base = frames[0]

    for df in frames[1:]:
        base = base.join(
            df,
            on=keys,
            how="full",
            coalesce=True,
        )

    exprs = []
    for source in SOURCE_TOPK:
        exprs += [
            pl.col(f"{source}_score").fill_null(0.0),
            pl.col(f"{source}_score_norm").fill_null(0.0),
            pl.col(f"{source}_rank").fill_null(9999),
            pl.col(f"{source}_rr").fill_null(0.0),
        ]

    base = base.with_columns(exprs)

    source_flags = [
        (pl.col(f"{source}_rank") < 9999).cast(pl.Int8)
        for source in SOURCE_TOPK
    ]

    base = (
        base
        .with_columns(
            pl.sum_horizontal(source_flags)
            .cast(pl.Int8)
            .alias("source_count")
        )
        .with_columns(
            (
                pl.col("candidate_item_id")
                == pl.col("target_item_id")
            ).cast(pl.Int8).alias("label")
        )
    )

    print(
        f"\n{split.upper()} | rows={base.height:,} | "
        f"cases={base['case_id'].n_unique():,} | "
        f"positives={int(base['label'].sum()):,}"
    )

    return base

train_fusion = build_candidate_fusion("train")
val_fusion = build_candidate_fusion("val")
test_fusion = build_candidate_fusion("test")

## 6. Candidate Recall

In [ ]:
def candidate_recall(df):
    per_case = (
        df.group_by("case_id")
        .agg(pl.col("label").max().alias("hit"))
    )
    return float(per_case["hit"].mean())

for name, df in [
    ("TRAIN", train_fusion),
    ("VAL", val_fusion),
    ("TEST", test_fusion),
]:
    print(name, "Candidate Recall =", round(candidate_recall(df), 6))

## 7. RRF Feature

In [ ]:
def add_rrf(df):
    terms = []

    for source in SOURCE_TOPK:
        terms.append(
            pl.when(pl.col(f"{source}_rank") < 9999)
            .then(1.0 / (60.0 + pl.col(f"{source}_rank")))
            .otherwise(0.0)
        )

    return df.with_columns(
        pl.sum_horizontal(terms)
        .cast(pl.Float32)
        .alias("rrf_score")
    )

train_fusion = add_rrf(train_fusion)
val_fusion = add_rrf(val_fusion)
test_fusion = add_rrf(test_fusion)

## 8. Item metadata + popularity từ TRAIN

In [ ]:
# Collect all unique candidate item IDs across splits to filter metadata
all_cand_ids = pl.concat([
    train_fusion.select(pl.col("candidate_item_id").cast(pl.Utf8).alias("item_id")),
    val_fusion.select(pl.col("candidate_item_id").cast(pl.Utf8).alias("item_id")),
    test_fusion.select(pl.col("candidate_item_id").cast(pl.Utf8).alias("item_id")),
]).unique()["item_id"].to_list()

print(f"Total unique candidate item IDs across all splits: {len(all_cand_ids):,}")

catalog_path = PROJECT_ROOT / "data" / "processed" / "recommender" / "serving" / "item_catalog_full.parquet"
cat_scan = pl.scan_parquet(catalog_path) if catalog_path.exists() else pl.scan_parquet(TRAIN_PATH)
cols = list(cat_scan.collect_schema().keys())

def first_existing(columns, choices, required=True):
    m = {c.lower(): c for c in columns}
    for x in choices:
        if x.lower() in m:
            return m[x.lower()]
    if required:
        raise KeyError(choices)
    return None

ITEM = first_existing(cols, ["item_id"])
PRODUCT = first_existing(cols, ["product_id"], False)
BRAND = first_existing(cols, ["brand_name", "brand"], False)
CAT0 = first_existing(cols, ["c0_name", "category0"], False)
CAT1 = first_existing(cols, ["c1_name", "category1"], False)
CAT2 = first_existing(cols, ["c2_name", "category2"], False)
PRICE = first_existing(cols, ["price"], False)
CONDITION = first_existing(cols, ["item_condition_name", "condition"], False)
SHIPPER = first_existing(cols, ["shipper_name", "shipper"], False)

print("Resolved:", ITEM, PRODUCT, BRAND, CAT0, CAT1, CAT2, PRICE, CONDITION, SHIPPER)


In [ ]:
def expr_or_default(col, alias, default):
    if col is None:
        return pl.lit(default).alias(alias)
    return pl.col(col).fill_null(default).alias(alias)

item_meta = (
    cat_scan
    .filter(pl.col(ITEM).cast(pl.Utf8).is_in(all_cand_ids))
    .select([
        pl.col(ITEM).cast(pl.Utf8).alias("candidate_item_id"),
        expr_or_default(PRODUCT, "candidate_product_id", "unknown"),
        expr_or_default(BRAND, "candidate_brand", "unknown"),
        expr_or_default(CAT0, "candidate_category0", "unknown"),
        expr_or_default(CAT1, "candidate_category1", "unknown"),
        expr_or_default(CAT2, "candidate_category2", "unknown"),
        expr_or_default(PRICE, "candidate_price", 0.0),
        expr_or_default(CONDITION, "candidate_condition", "unknown"),
        expr_or_default(SHIPPER, "candidate_shipper", "unknown"),
    ])
    .collect(engine="streaming")
    .unique(subset=["candidate_item_id"], keep="last")
)

popularity = (
    pl.scan_parquet(TRAIN_PATH)
    .filter(pl.col(ITEM).cast(pl.Utf8).is_in(all_cand_ids))
    .group_by(pl.col(ITEM).cast(pl.Utf8).alias("candidate_item_id"))
    .agg(pl.len().cast(pl.Float32).alias("candidate_popularity"))
    .collect(engine="streaming")
)

item_meta = item_meta.join(popularity, on="candidate_item_id", how="left").select([
    "candidate_item_id", "candidate_product_id", "candidate_brand",
    "candidate_category0", "candidate_category1", "candidate_category2",
    "candidate_price", "candidate_condition", "candidate_shipper",
    pl.col("candidate_popularity").fill_null(0.0)
])

print("Item metadata:", f"{item_meta.height:,}")


In [ ]:
def join_item_features(df):
    return (
        df
        .join(item_meta, on="candidate_item_id", how="left")
        .with_columns([
            pl.col("candidate_product_id").fill_null("unknown"),
            pl.col("candidate_brand").fill_null("unknown"),
            pl.col("candidate_category0").fill_null("unknown"),
            pl.col("candidate_category1").fill_null("unknown"),
            pl.col("candidate_category2").fill_null("unknown"),
            pl.col("candidate_condition").fill_null("unknown"),
            pl.col("candidate_shipper").fill_null("unknown"),
            pl.col("candidate_price").cast(pl.Float32, strict=False).fill_null(0.0),
            pl.col("candidate_popularity").cast(pl.Float32, strict=False).fill_null(0.0),
        ])
    )

train_fusion = join_item_features(train_fusion)
val_fusion = join_item_features(val_fusion)
test_fusion = join_item_features(test_fusion)

## 9. Hard-negative sampling cho TRAIN

Mỗi case:
- giữ positive
- giữ tối đa 199 hard negatives có RRF cao nhất

In [ ]:
TRAIN_NEGATIVES = 199

positive_train = train_fusion.filter(pl.col("label") == 1)

negative_train = (
    train_fusion
    .filter(pl.col("label") == 0)
    .sort(["case_id", "rrf_score"], descending=[False, True])
    .group_by("case_id", maintain_order=True)
    .head(TRAIN_NEGATIVES)
)

train_rank = pl.concat(
    [positive_train, negative_train],
    how="vertical_relaxed",
)

print("TRAIN ranking rows:", f"{train_rank.height:,}")
print("Positives:", f"{int(train_rank['label'].sum()):,}")

## 10. Feature columns

In [ ]:
CAT_COLUMNS = [
    "user_id",
    "candidate_item_id",
    "candidate_product_id",
    "candidate_brand",
    "candidate_category0",
    "candidate_category1",
    "candidate_category2",
    "candidate_condition",
    "candidate_shipper",
]

NUM_COLUMNS = [
    "source_count",
    "rrf_score",
    "candidate_price",
    "candidate_popularity",
]

for source in SOURCE_TOPK:
    NUM_COLUMNS += [
        f"{source}_score",
        f"{source}_score_norm",
        f"{source}_rank",
        f"{source}_rr",
    ]

print("Categorical =", len(CAT_COLUMNS))
print("Numerical   =", len(NUM_COLUMNS))

## 11. Save DCN datasets

In [ ]:
DCN_TRAIN = DATASET_DIR / "dcn_train.parquet"
DCN_VAL = DATASET_DIR / "dcn_val.parquet"
DCN_TEST = DATASET_DIR / "dcn_test.parquet"

train_rank.write_parquet(DCN_TRAIN, compression="zstd")
val_fusion.write_parquet(DCN_VAL, compression="zstd")
test_fusion.write_parquet(DCN_TEST, compression="zstd")

print(DCN_TRAIN)
print(DCN_VAL)
print(DCN_TEST)

## 12. Hash embedding config

In [ ]:
HASH_BUCKETS = {
    "user_id": 50_000,
    "candidate_item_id": 100_000,
    "candidate_product_id": 50_000,
    "candidate_brand": 10_000,
    "candidate_category0": 2_000,
    "candidate_category1": 5_000,
    "candidate_category2": 10_000,
    "candidate_condition": 512,
    "candidate_shipper": 512,
}


## 13. Load datasets

In [ ]:
train_pl = pl.read_parquet(DCN_TRAIN)
val_pl = pl.read_parquet(DCN_VAL)
test_pl = pl.read_parquet(DCN_TEST)

print("TRAIN:", train_pl.shape)
print("VAL  :", val_pl.shape)
print("TEST :", test_pl.shape)


## 14. Numerical preprocessing

In [ ]:
def preprocess_numerical(df):
    exprs = []
    for source in SOURCE_TOPK:
        rank_col = f"{source}_rank"
        exprs.append((1.0 / (pl.col(rank_col).cast(pl.Float32) + 1.0)).alias(rank_col))
    exprs.append((pl.col("candidate_price").cast(pl.Float32).fill_null(0.0).clip(0, 1e8) + 1.0).log().alias("candidate_price"))
    exprs.append((pl.col("candidate_popularity").cast(pl.Float32).fill_null(0.0).clip(0, 1e8) + 1.0).log().alias("candidate_popularity"))
    return df.with_columns(exprs)

train_pl = preprocess_numerical(train_pl)
val_pl = preprocess_numerical(val_pl)
test_pl = preprocess_numerical(test_pl)

num_np = train_pl.select(NUM_COLUMNS).to_numpy().astype(np.float32)
num_mean = num_np.mean(axis=0)
num_std = num_np.std(axis=0)
num_std[(~np.isfinite(num_std)) | (num_std < 1e-6)] = 1.0

print(num_mean.shape, num_std.shape)


## 15. PyTorch Dataset

In [ ]:
def get_hashed_cats(df):
    exprs = []
    for col in CAT_COLUMNS:
        b = HASH_BUCKETS[col]
        exprs.append((pl.col(col).cast(pl.Utf8).fill_null("unknown").hash(seed=42) % (b - 1) + 1).cast(pl.Int64).alias(f"{col}_hash"))
    hashed_df = df.with_columns(exprs)
    cat_cols = [f"{col}_hash" for col in CAT_COLUMNS]
    return hashed_df.select(cat_cols).to_numpy()

class DCNDataset(Dataset):
    def __init__(self, df, num_mean, num_std):
        nums = df.select(NUM_COLUMNS).to_numpy().astype(np.float32)
        self.nums = (nums - num_mean) / num_std
        self.cats = get_hashed_cats(df)
        self.labels = df["label"].to_numpy().astype(np.float32)
        # Integer categorical encoding of case_id
        self.case_ids = df.select(pl.col("case_id").to_physical()).to_numpy().squeeze(-1).astype(np.int64)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return (
            torch.from_numpy(self.cats[idx]),
            torch.from_numpy(self.nums[idx]),
            torch.tensor(self.labels[idx], dtype=torch.float32),
            torch.tensor(self.case_ids[idx], dtype=torch.long),
        )

train_ds = DCNDataset(train_pl, num_mean, num_std)
val_ds = DCNDataset(val_pl, num_mean, num_std)
test_ds = DCNDataset(test_pl, num_mean, num_std)

print(len(train_ds), len(val_ds), len(test_ds))


## 16. DCN Model

In [ ]:
class CrossLayer(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.weight = nn.Parameter(torch.randn(dim) * 0.01)
        self.bias = nn.Parameter(torch.zeros(dim))

    def forward(self, x0, x):
        dot = torch.sum(x * self.weight, dim=1, keepdim=True)
        return x0 * dot + self.bias + x


class DCN(nn.Module):
    def __init__(
        self,
        hash_buckets,
        num_numeric,
        embed_dim=8,
        cross_layers=3,
        deep_layers=(256, 128, 64),
        dropout=0.15,
    ):
        super().__init__()

        self.cat_columns = list(hash_buckets.keys())

        self.embeddings = nn.ModuleDict({
            col: nn.Embedding(buckets, embed_dim)
            for col, buckets in hash_buckets.items()
        })

        input_dim = (
            len(self.cat_columns) * embed_dim
            + num_numeric
        )

        self.cross_layers = nn.ModuleList([
            CrossLayer(input_dim)
            for _ in range(cross_layers)
        ])

        layers = []
        dim = input_dim

        for hidden in deep_layers:
            layers += [
                nn.Linear(dim, hidden),
                nn.ReLU(),
                nn.LayerNorm(hidden),
                nn.Dropout(dropout),
            ]
            dim = hidden

        self.deep = nn.Sequential(*layers)
        self.output = nn.Linear(input_dim + dim, 1)

    def forward(self, cats, nums):
        embs = [
            self.embeddings[col](cats[:, i])
            for i, col in enumerate(self.cat_columns)
        ]

        x0 = torch.cat(embs + [nums], dim=1)

        cross = x0
        for layer in self.cross_layers:
            cross = layer(x0, cross)

        deep = self.deep(x0)

        return self.output(
            torch.cat([cross, deep], dim=1)
        ).squeeze(1)

In [ ]:
model = DCN(
    HASH_BUCKETS,
    num_numeric=len(NUM_COLUMNS),
    embed_dim=8,
    cross_layers=3,
    deep_layers=(256, 128, 64),
    dropout=0.15,
).to(DEVICE)

params = sum(p.numel() for p in model.parameters())

print(model)
print("\nParameters:", f"{params:,}")

## 17. DataLoader + Loss

In [ ]:
BATCH_SIZE = 2048

train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
)

val_loader = DataLoader(
    val_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
)

test_loader = DataLoader(
    test_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-3,
    weight_decay=1e-5,
)

positive_count = float(train_pl["label"].sum())
negative_count = float(len(train_pl) - positive_count)

pos_weight = min(
    negative_count / max(positive_count, 1.0),
    20.0,
)

criterion = nn.BCEWithLogitsLoss(
    pos_weight=torch.tensor(pos_weight, dtype=torch.float32).to(DEVICE)
)

print("positive:", int(positive_count))
print("negative:", int(negative_count))
print("pos_weight:", pos_weight)


## 18. Ranking metrics

In [ ]:
@torch.no_grad()
def predict_loader(model, loader):
    model.eval()

    scores_all = []
    labels_all = []
    cases_all = []

    for cats, nums, labels, case_ids in loader:
        logits = model(
            cats.to(DEVICE),
            nums.to(DEVICE),
        )

        scores_all.append(
            torch.sigmoid(logits).cpu().numpy()
        )

        labels_all.append(labels.numpy())
        cases_all.append(case_ids.numpy())

    return (
        np.concatenate(scores_all),
        np.concatenate(labels_all),
        np.concatenate(cases_all),
    )


def ranking_metrics(scores, labels, case_ids, ks=(10, 20, 50, 100)):
    df = pd.DataFrame({
        "case_id": case_ids,
        "score": scores,
        "label": labels,
    })

    recalls = {k: [] for k in ks}
    ndcgs = {k: [] for k in ks}
    mrr20 = []

    for _, g in df.groupby("case_id"):
        if g["label"].sum() <= 0:
            continue

        g = g.sort_values("score", ascending=False)
        ranked = g["label"].values

        pos = np.where(ranked > 0)[0]
        if len(pos) == 0:
            continue

        rank = int(pos[0]) + 1

        for k in ks:
            hit = rank <= k
            recalls[k].append(float(hit))
            ndcgs[k].append(
                1.0 / math.log2(rank + 1)
                if hit
                else 0.0
            )

        mrr20.append(
            1.0 / rank if rank <= 20 else 0.0
        )

    out = {}

    for k in ks:
        out[f"Recall@{k}"] = float(np.mean(recalls[k])) if recalls[k] else 0.0
        out[f"NDCG@{k}"] = float(np.mean(ndcgs[k])) if ndcgs[k] else 0.0

    out["MRR@20"] = float(np.mean(mrr20)) if mrr20 else 0.0

    return out


def evaluate(model, loader):
    scores, labels, cases = predict_loader(model, loader)
    result = ranking_metrics(scores, labels, cases)

    try:
        result["AUC"] = float(
            roc_auc_score(labels, scores)
        )
    except Exception:
        result["AUC"] = None

    return result

## 19. Train DCN

In [ ]:
EPOCHS = 10
PATIENCE = 3

BEST_PATH = MODEL_DIR / "dcn_best.pt"

history = []
best_ndcg = -1.0
bad_epochs = 0

for epoch in range(1, EPOCHS + 1):
    model.train()

    total_loss = 0.0
    batches = 0

    for cats, nums, labels, _ in train_loader:
        cats = cats.to(DEVICE)
        nums = nums.to(DEVICE)
        labels = labels.to(DEVICE)

        optimizer.zero_grad()

        logits = model(cats, nums)
        loss = criterion(logits, labels)

        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            5.0,
        )

        optimizer.step()

        total_loss += loss.item()
        batches += 1

    train_loss = total_loss / max(batches, 1)

    val_metrics = evaluate(model, val_loader)

    row = {
        "epoch": epoch,
        "train_loss": train_loss,
        **val_metrics,
    }

    history.append(row)

    print("\n" + "=" * 70)
    print("EPOCH", epoch)
    print("Train loss:", train_loss)
    print(json.dumps(val_metrics, indent=2))

    ndcg20 = val_metrics["NDCG@20"]

    if ndcg20 > best_ndcg:
        best_ndcg = ndcg20
        bad_epochs = 0

        torch.save(
            {
                "model": model.state_dict(),
                "epoch": epoch,
                "best_ndcg20": best_ndcg,
                "num_mean": num_mean,
                "num_std": num_std,
                "cat_columns": CAT_COLUMNS,
                "num_columns": NUM_COLUMNS,
                "hash_buckets": HASH_BUCKETS,
                "source_topk": SOURCE_TOPK,
            },
            BEST_PATH,
        )

        print("[BEST] NEW BEST:", best_ndcg)

    else:
        bad_epochs += 1
        print("No improvement:", bad_epochs, "/", PATIENCE)

    if bad_epochs >= PATIENCE:
        print("EARLY STOPPING")
        break

with open(
    METRIC_DIR / "train_history.json",
    "w",
    encoding="utf-8",
) as f:
    json.dump(history, f, indent=2)

## 20. Final TEST

In [ ]:
checkpoint = torch.load(
    BEST_PATH,
    map_location="cpu",
    weights_only=False,
)

model.load_state_dict(
    checkpoint["model"]
)

test_metrics = evaluate(
    model,
    test_loader,
)

print("Best epoch:", checkpoint["epoch"])
print("Best VAL NDCG@20:", checkpoint["best_ndcg20"])
print(json.dumps(test_metrics, indent=2))

with open(
    METRIC_DIR / "test_metrics.json",
    "w",
    encoding="utf-8",
) as f:
    json.dump(test_metrics, f, indent=2)

summary = {
    "model": "DCN Ranker",
    "source_topk": SOURCE_TOPK,
    "train_rows": len(train_pl),
    "val_rows": len(val_pl),
    "test_rows": len(test_pl),
    "val_candidate_recall": candidate_recall(val_fusion),
    "test_candidate_recall": candidate_recall(test_fusion),
    "best_epoch": checkpoint["epoch"],
    "best_val_ndcg20": checkpoint["best_ndcg20"],
    "test_metrics": test_metrics,
}

with open(
    METRIC_DIR / "summary.json",
    "w",
    encoding="utf-8",
) as f:
    json.dump(summary, f, indent=2)

print("\nSUMMARY")
print(json.dumps(summary, indent=2))


# Output

Sau khi chạy xong:

`D:\MerRec\training\dcn`

sẽ có:

- `dataset/dcn_train.parquet`
- `dataset/dcn_val.parquet`
- `dataset/dcn_test.parquet`
- `model/dcn_best.pt`
- `metrics/train_history.json`
- `metrics/test_metrics.json`
- `metrics/summary.json`